# Endotoxin assay plate setup

This notebook prepares the 0.005-5 EU/mL endotoxin standards and loads 30 wells of a black Corning 360 µL flat-bottom plate. It uses McKesson 12 × 75 mm tubes in a bare Hamilton 32-position tube carrier, following the calibrated geometry from `../testing-dimensions/12x75mm_tube_height_test.ipynb`.

> **Intentional protocol deviation:** working solution is dispensed before samples, standards, and EFW, as requested. The kit guide instead recommends adding samples/standards, preheating for 10 minutes at 37 °C, and adding working solution last. Validate this deviation before using assay results.

Run cells from top to bottom once. Confirm the printed deck and volume plan before setup. Do not rerun deck construction or a partially completed stage. All tips and direct-contact labware must be endotoxin-free.


In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
from collections import defaultdict
from copy import deepcopy

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend, LiquidHandlerChatterboxBackend
from pylabrobot.liquid_handling.standard import Mix
from pylabrobot.resources import (
    McKesson_tube_5mL_Rb,
    cor_96_wellplate_360uL_Fb,
    hamilton_96_tiprack_1000uL_filter,
    hamilton_96_tiprack_300uL_filter,
    hamilton_96_tiprack_50uL_filter,
    set_volume_tracking,
)
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_mfx_carrier_L5_base,
    hamilton_tube_carrier_32_a00,
)
from pylabrobot.resources.hamilton.mfx_modules import (
    hamilton_mfx_plateholder_DWP_metal_tapped,
)


## Configuration and protocol plan

Set `SIMULATE = True` only for a no-hardware dry run. PyLabRobot channel indices are zero-based; physical channel 1 is index 0 and physical channels 5-6 are indices 4-5. Physical tube position 32 is the frontmost carrier site and maps to PyLabRobot site 0.


In [ ]:
SIMULATE = False
SINGLE_CHANNEL = 0
WORKING_CHANNELS = [4, 5]
TUBE_TOP_ABOVE_CARRIER_MM = 25.6
TUBE_MATERIAL_Z_THICKNESS_MM = 1.0
TUBE_LLD_SEARCH_HEIGHT_MM = 75.0  # Relative to the tube's inner bottom; near its rim.
TUBE_ASPIRATE_HEIGHT_MM = 6.0
PLATE_SOURCE_ASPIRATE_HEIGHT_MM = 3.0
TUBE_DISPENSE_HEIGHT_MM = 12.0
STANDARD_MIX_VOLUME_UL = 900.0
STANDARD_MIX_REPETITIONS = 2
STANDARD_MIX_FLOW_UL_S = 400.0
# STANDARD_MIX_CYCLES = 4
STANDARD_MIX_CYCLES = 1
STANDARD_PAUSE_SECONDS = 0.0 if SIMULATE else 20.0
set_volume_tracking(enabled=True)

def physical_to_site(position: int) -> int:
    if not 1 <= position <= 32:
        raise ValueError(f"Physical tube position must be 1-32, got {position}.")
    return 32 - position

RAIL19_TUBES = {
    "efw_standard": {"position": 32, "initial_ul": 4000.0},
    "efw_plate": {"position": 31, "initial_ul": 4000.0},
    "efw_reserve_1": {"position": 30, "initial_ul": 4000.0},
    "efw_reserve_2": {"position": 29, "initial_ul": 4000.0},
    "std_0_005": {"position": 28, "initial_ul": 0.0, "eu_ml": 0.005},
    "std_0_05": {"position": 27, "initial_ul": 0.0, "eu_ml": 0.05},
    "std_0_5": {"position": 26, "initial_ul": 0.0, "eu_ml": 0.5},
    "std_5": {"position": 25, "initial_ul": 0.0, "eu_ml": 5.0},
    "working_1": {"position": 23, "initial_ul": 2000.0},
    "working_2": {"position": 22, "initial_ul": 2000.0},
    "sample_1": {"position": 20, "initial_ul": 1000.0},
    "sample_2": {"position": 19, "initial_ul": 1000.0},
}
STANDARD_PREP = [
    {"key": "std_5", "efw_ul": 750.0, "stock_ul": 250.0, "mix_tip": "B1"},
    {"key": "std_0_5", "efw_ul": 900.0, "from": "std_5", "transfer_ul": 100.0, "transfer_tip": "A1", "mix_tip": "C1"},
    {"key": "std_0_05", "efw_ul": 900.0, "from": "std_0_5", "transfer_ul": 100.0, "transfer_tip": "B1", "mix_tip": "D1"},
    {"key": "std_0_005", "efw_ul": 900.0, "from": "std_0_05", "transfer_ul": 100.0, "transfer_tip": "C1", "mix_tip": "E1"},
]
WORKING_WELLS = [f"A{i}" for i in range(1, 13)] + [f"B{i}" for i in range(1, 13)] + ["C1", "C2", "D1", "D2", "E1", "E2"]

PLATE_TRANSFER_GROUPS = [
    {"source": "efw_plate", "tip_size": 1000, "tip": "H1", "batch_aspirate": True, "reserve_ul": 200.0, "destinations": [
        ("A1", 100.0), ("A2", 100.0), ("A7", 75.0), ("A8", 75.0),
    ]},
    {"source": "efw_plate", "tip_size": 1000, "tip": "A2", "batch_aspirate": True, "reserve_ul": 200.0, "destinations": [
        ("B7", 75.0), ("B8", 75.0), ("A9", 87.5), ("B9", 87.5),
    ]},
    {"source": "efw_plate", "tip_size": 1000, "tip": "B2", "batch_aspirate": True, "reserve_ul": 200.0, "destinations": [
        ("A10", 93.75), ("B10", 93.75),
    ]},
    {"source": "efw_plate", "tip_size": 1000, "tip": "C2", "batch_aspirate": True, "reserve_ul": 200.0, "destinations": [("A5", 50.0), ("A6", 50.0), ("B5", 50.0), ("B6", 50.0)]},
    {"source": "sample_1", "tip_size": 300, "tip": "D1", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("A3", 100.0), ("A4", 100.0)]},
    {"source": "sample_2", "tip_size": 300, "tip": "E1", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("B3", 100.0), ("B4", 100.0)]},
    {"source": "std_0_005", "tip_size": 300, "tip": "F1", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("B1", 100.0), ("B2", 100.0)]},
    {"source": "std_0_05", "tip_size": 300, "tip": "G1", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("C1", 100.0), ("C2", 100.0)]},
    {"source": "std_0_5", "tip_size": 300, "tip": "H1", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("D1", 100.0), ("D2", 100.0)]},
    {"source": "std_5", "tip_size": 300, "tip": "A2", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("E1", 100.0), ("E2", 100.0)]},
    {"source": "sample_1", "tip_size": 300, "tip": "B2", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("A5", 50.0), ("A6", 50.0), ("A11", 50.0), ("A12", 50.0)]},
    {"source": "sample_2", "tip_size": 300, "tip": "C2", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("B5", 50.0), ("B6", 50.0), ("B11", 50.0), ("B12", 50.0)]},
    {"source": "std_0_5", "tip_size": 300, "tip": "D2", "batch_aspirate": True, "reserve_ul": 50.0, "destinations": [("A11", 50.0), ("A12", 50.0), ("B11", 50.0), ("B12", 50.0)]},
    {"source": "sample_1", "tip_size": 50, "tip": "A1", "batch_aspirate": True, "reserve_ul": 15.0, "destinations": [("A7", 25.0)]},
    {"source": "sample_1", "tip_size": 50, "tip": "B1", "batch_aspirate": True, "reserve_ul": 15.0, "destinations": [("A8", 25.0)]},
    {"source": "sample_1", "tip_size": 50, "tip": "C1", "batch_aspirate": True, "reserve_ul": 11.25, "destinations": [("A9", 12.5), ("A10", 6.25)]},
    {"source": "sample_2", "tip_size": 50, "tip": "D1", "batch_aspirate": True, "reserve_ul": 15.0, "destinations": [("B7", 25.0)]},
    {"source": "sample_2", "tip_size": 50, "tip": "E1", "batch_aspirate": True, "reserve_ul": 15.0, "destinations": [("B8", 25.0)]},
    {"source": "sample_2", "tip_size": 50, "tip": "F1", "batch_aspirate": True, "reserve_ul": 11.25, "destinations": [("B9", 12.5), ("B10", 6.25)]},
]
PLATE_LABELS = {
    "A1": "Blank", "A2": "Blank", "A3": "sample 1 1:1", "A4": "sample 1 1:1",
    "A5": "sample 1 1:2", "A6": "sample 1 1:2", "A7": "sample 1 1:4", "A8": "sample 1 1:4",
    "A9": "sample 1 1:8", "A10": "sample 1 1:32", "A11": "sample 1 SPIKED 1:2", "A12": "sample 1 SPIKED 1:2",
    "B1": "Std1 0.005 EU/mL", "B2": "Std1 0.005 EU/mL", "B3": "sample 2 1:1", "B4": "sample 2 1:1",
    "B5": "sample 2 1:2", "B6": "sample 2 1:2", "B7": "sample 2 1:4", "B8": "sample 2 1:4",
    "B9": "sample 2 1:8", "B10": "sample 2 1:32", "B11": "sample 2 SPIKED 1:2", "B12": "sample 2 SPIKED 1:2",
    "C1": "Std2 0.05 EU/mL", "C2": "Std2 0.05 EU/mL", "D1": "Std3 0.5 EU/mL", "D2": "Std3 0.5 EU/mL",
    "E1": "Std4 5 EU/mL", "E2": "Std4 5 EU/mL",
}


## Pure preflight validation

This validation performs no motion. It verifies the corrected plate map, source residuals, tip policy, serial dilution routing, and carrier position conversion.


In [ ]:
def required_tip_size(volume_ul: float) -> int:
    if 0.0 < volume_ul <= 50.0:
        return 50
    if volume_ul <= 300.0:
        return 300
    if volume_ul > 300.0:
        return 1000
    raise ValueError(f"Transfer volume must be positive, got {volume_ul} µL")

def validate_protocol_plan():
    assert physical_to_site(32) == 0 and physical_to_site(31) == 1
    assert physical_to_site(28) == 4 and physical_to_site(22) == 10
    assert physical_to_site(20) == 12 and physical_to_site(19) == 13
    assert 24 not in {cfg["position"] for cfg in RAIL19_TUBES.values()}
    assert len(RAIL19_TUBES) == 12
    assert RAIL19_TUBES["sample_1"]["position"] == 20
    assert RAIL19_TUBES["sample_2"]["position"] == 19
    assert required_tip_size(50.0) == 50
    assert required_tip_size(50.01) == required_tip_size(300.0) == 300
    assert required_tip_size(300.01) == 1000
    assert [RAIL19_TUBES[s["key"]]["eu_ml"] for s in STANDARD_PREP] == [5.0, 0.5, 0.05, 0.005]
    assert all(s["efw_ul"] == 900.0 for s in STANDARD_PREP[1:])
    assert STANDARD_PREP[0]["efw_ul"] == 750.0 and STANDARD_PREP[0]["stock_ul"] == 250.0
    assert all(required_tip_size(s["efw_ul"]) == 1000 for s in STANDARD_PREP)
    assert all(required_tip_size(s["transfer_ul"]) == 300 for s in STANDARD_PREP[1:])
    assert [s["transfer_tip"] for s in STANDARD_PREP[1:]] == ["A1", "B1", "C1"]
    assert required_tip_size(STANDARD_MIX_VOLUME_UL) == 1000

    plate_additions = defaultdict(float)
    source_use = defaultdict(float)
    for group in PLATE_TRANSFER_GROUPS:
        group_total_ul = sum(volume_ul for _, volume_ul in group["destinations"])
        aspiration_ul = group_total_ul + group["reserve_ul"]
        assert group.get("batch_aspirate", False)
        assert group["reserve_ul"] > 0.0
        assert aspiration_ul <= group["tip_size"]
        assert len(group["destinations"]) <= 4
        if group["tip_size"] == 1000:
            assert group["source"] == "efw_plate" and group["reserve_ul"] >= 200.0
            assert aspiration_ul <= 700.0
        elif group["tip_size"] == 300:
            assert group["reserve_ul"] >= 50.0 and aspiration_ul == 250.0
        else:
            assert group["tip_size"] == 50 and aspiration_ul in {30.0, 40.0}
            assert all(volume_ul < 50.0 for _, volume_ul in group["destinations"])
        source_use[group["source"]] += group["reserve_ul"]
        for well_name, volume_ul in group["destinations"]:
            plate_additions[well_name] += volume_ul
            source_use[group["source"]] += volume_ul
    tips_by_size = defaultdict(list)
    for group in PLATE_TRANSFER_GROUPS:
        tips_by_size[group["tip_size"]].append(group["tip"])
    assert all(len(tips) == len(set(tips)) for tips in tips_by_size.values())
    assert tips_by_size[1000] == ["H1", "A2", "B2", "C2"]
    assert tips_by_size[300] == ["D1", "E1", "F1", "G1", "H1", "A2", "B2", "C2", "D2"]
    assert tips_by_size[50] == ["A1", "B1", "C1", "D1", "E1", "F1"]
    assert set(WORKING_WELLS) == set(PLATE_LABELS) == set(plate_additions)
    assert len(WORKING_WELLS) == 30 and len(set(WORKING_WELLS)) == 30
    assert all(abs(plate_additions[w] - 100.0) < 1e-9 for w in WORKING_WELLS)
    assert all(100.0 + plate_additions[w] == 200.0 for w in WORKING_WELLS)
    assert source_use["efw_plate"] == 1862.5
    assert source_use["sample_1"] == source_use["sample_2"] == 610.0
    assert source_use["std_0_005"] == source_use["std_0_05"] == source_use["std_5"] == 250.0
    assert source_use["std_0_5"] == 500.0
    assert sum(volume for group in PLATE_TRANSFER_GROUPS if group["source"] == "std_0_5" for _, volume in group["destinations"]) == 400.0

    expected_residuals = {
        "efw_standard": 550.0, "efw_plate": 2137.5,
        "efw_reserve_1": 4000.0, "efw_reserve_2": 4000.0,
        "working_1": 500.0, "working_2": 500.0,
        "sample_1": 390.0, "sample_2": 390.0,
        "std_5": 650.0, "std_0_5": 400.0,
        "std_0_05": 650.0, "std_0_005": 750.0,
    }
    assert max(200.0 for _ in WORKING_WELLS) <= 360.0
    return {
        "plate_final_volumes_ul": {w: 200.0 for w in WORKING_WELLS},
        "plate_labels": deepcopy(PLATE_LABELS),
        "source_use_ul": dict(source_use),
        "expected_residuals_ul": expected_residuals,
    }

PLAN_CHECK = validate_protocol_plan()
print("Protocol plan validated. Expected source residuals (µL):")
for source, volume in PLAN_CHECK["expected_residuals_ul"].items():
    print(f"  {source:16s} {volume:8.2f}")


## Expected vs. programmed well-volume audit

This preflight-only report compares the programmed liquid additions with the expected 30-well assay layout. Tip reserves are excluded because they are discarded and never enter the plate. A zero-delta plan verifies requested software volumes, not physical delivery.


In [ ]:
VOLUME_COLUMNS = [
    "Number", "Well", "Label", "WorkingSoln", "EFW", "Std1", "Std2",
    "Std3", "Std4", "sample1", "sample2", "SampleDilutionFactor",
    "TOTALVOL(ul)",
]
COMPONENT_COLUMNS = [
    "WorkingSoln", "EFW", "Std1", "Std2", "Std3", "Std4",
    "sample1", "sample2",
]
EXPECTED_VOLUME_ROWS = [
    (1, "A1", "Blank", 100, 100, 0, 0, 0, 0, 0, 0, "", 200),
    (2, "A2", "Blank", 100, 100, 0, 0, 0, 0, 0, 0, "", 200),
    (3, "A3", "sample 1\n1:1", 100, 0, 0, 0, 0, 0, 100, 0, "1:2", 200),
    (4, "A4", "sample 1\n1:1", 100, 0, 0, 0, 0, 0, 100, 0, "1:2", 200),
    (5, "A5", "sample 1\n1:2", 100, 50, 0, 0, 0, 0, 50, 0, "1:4", 200),
    (6, "A6", "sample 1\n1:2", 100, 50, 0, 0, 0, 0, 50, 0, "1:4", 200),
    (7, "A7", "sample 1\n1:4", 100, 75, 0, 0, 0, 0, 25, 0, "1:8", 200),
    (8, "A8", "sample 1\n1:4", 100, 75, 0, 0, 0, 0, 25, 0, "1:8", 200),
    (9, "A9", "sample 1\n1:8", 100, 87.5, 0, 0, 0, 0, 12.5, 0, "1:16", 200),
    (10, "A10", "sample 1\n1:32", 100, 93.75, 0, 0, 0, 0, 6.25, 0, "1:32", 200),
    (11, "A11", "sample 1\nSPIKED\n1:2", 100, 0, 0, 0, 50, 0, 50, 0, "", 200),
    (12, "A12", "sample 1\nSPIKED\n1:2", 100, 0, 0, 0, 50, 0, 50, 0, "", 200),
    (13, "B1", "Std1 0.005", 100, 0, 100, 0, 0, 0, 0, 0, "", 200),
    (14, "B2", "Std1", 100, 0, 100, 0, 0, 0, 0, 0, "", 200),
    (15, "B3", "sample 2\n1:1", 100, 0, 0, 0, 0, 0, 0, 100, "1:2", 200),
    (16, "B4", "sample 2\n1:1", 100, 0, 0, 0, 0, 0, 0, 100, "1:2", 200),
    (17, "B5", "sample 2\n1:2", 100, 50, 0, 0, 0, 0, 0, 50, "1:4", 200),
    (18, "B6", "sample 2\n1:2", 100, 50, 0, 0, 0, 0, 0, 50, "1:4", 200),
    (19, "B7", "sample 2\n1:4", 100, 75, 0, 0, 0, 0, 0, 25, "1:8", 200),
    (20, "B8", "sample 2\n1:4", 100, 75, 0, 0, 0, 0, 0, 25, "1:8", 200),
    (21, "B9", "sample 2\n1:8", 100, 87.5, 0, 0, 0, 0, 0, 12.5, "1:16", 200),
    (22, "B10", "sample 2\n1:32", 100, 93.75, 0, 0, 0, 0, 0, 6.25, "1:32", 200),
    (23, "B11", "sample 2\nSPIKED\n1:2", 100, 0, 0, 0, 50, 0, 0, 50, "", 200),
    (24, "B12", "sample 2\nSPIKED\n1:2", 100, 0, 0, 0, 50, 0, 0, 50, "", 200),
    (25, "C1", "Std2 0.05", 100, 0, 0, 100, 0, 0, 0, 0, "", 200),
    (26, "C2", "Std2", 100, 0, 0, 100, 0, 0, 0, 0, "", 200),
    (27, "D1", "Std3 0.5", 100, 0, 0, 0, 100, 0, 0, 0, "", 200),
    (28, "D2", "Std3", 100, 0, 0, 0, 100, 0, 0, 0, "", 200),
    (29, "E1", "Std4 5", 100, 0, 0, 0, 0, 100, 0, 0, "", 200),
    (30, "E2", "Std4", 100, 0, 0, 0, 0, 100, 0, 0, "", 200),
]
EXPECTED_VOLUME_DF = pd.DataFrame(EXPECTED_VOLUME_ROWS, columns=VOLUME_COLUMNS)

SOURCE_TO_COMPONENT = {
    "efw_plate": "EFW", "std_0_005": "Std1", "std_0_05": "Std2",
    "std_0_5": "Std3", "std_5": "Std4",
    "sample_1": "sample1", "sample_2": "sample2",
}
programmed_components = {
    well: {component: 0.0 for component in COMPONENT_COLUMNS}
    for well in WORKING_WELLS
}
for well in WORKING_WELLS:
    programmed_components[well]["WorkingSoln"] = 100.0
for group in PLATE_TRANSFER_GROUPS:
    component = SOURCE_TO_COMPONENT[group["source"]]
    for well, volume_ul in group["destinations"]:
        programmed_components[well][component] += volume_ul

programmed_rows = []
for expected in EXPECTED_VOLUME_DF.to_dict(orient="records"):
    well = expected["Well"]
    row = {
        "Number": expected["Number"], "Well": well, "Label": expected["Label"],
        **programmed_components[well],
        "SampleDilutionFactor": expected["SampleDilutionFactor"],
    }
    row["TOTALVOL(ul)"] = sum(row[column] for column in COMPONENT_COLUMNS)
    programmed_rows.append(row)
PROGRAMMED_VOLUME_DF = pd.DataFrame(programmed_rows, columns=VOLUME_COLUMNS)

comparison_rows = []
for expected, programmed in zip(
    EXPECTED_VOLUME_DF.to_dict(orient="records"),
    PROGRAMMED_VOLUME_DF.to_dict(orient="records"),
):
    comparison = {"Number": expected["Number"], "Well": expected["Well"], "Label": expected["Label"]}
    for component in COMPONENT_COLUMNS:
        comparison[f"Delta_{component}"] = programmed[component] - expected[component]
    comparison["ProgrammedTOTAL"] = programmed["TOTALVOL(ul)"]
    comparison["ExpectedTOTAL"] = expected["TOTALVOL(ul)"]
    comparison["TotalDelta"] = comparison["ProgrammedTOTAL"] - comparison["ExpectedTOTAL"]
    comparison["Match"] = all(abs(comparison[f"Delta_{c}"]) <= 1e-9 for c in COMPONENT_COLUMNS) and abs(comparison["TotalDelta"]) <= 1e-9
    comparison_rows.append(comparison)
VOLUME_COMPARISON_DF = pd.DataFrame(comparison_rows)

assert list(EXPECTED_VOLUME_DF.columns) == VOLUME_COLUMNS
assert len(EXPECTED_VOLUME_DF) == 30 and EXPECTED_VOLUME_DF["Well"].is_unique
assert set(EXPECTED_VOLUME_DF["Well"]) == set(WORKING_WELLS)
assert (EXPECTED_VOLUME_DF["TOTALVOL(ul)"] == 200).all()
assert (PROGRAMMED_VOLUME_DF["TOTALVOL(ul)"] == 200).all()
assert VOLUME_COMPARISON_DF["Match"].all()
scheduled_plate_ul = sum(volume_ul for group in PLATE_TRANSFER_GROUPS for _, volume_ul in group["destinations"])
discarded_reserve_ul = sum(group["reserve_ul"] for group in PLATE_TRANSFER_GROUPS)
programmed_nonworking_ul = PROGRAMMED_VOLUME_DF[[c for c in COMPONENT_COLUMNS if c != "WorkingSoln"]].to_numpy().sum()
assert programmed_nonworking_ul == scheduled_plate_ul and discarded_reserve_ul > 0.0
assert PLATE_LABELS["A10"].endswith("1:32") and PLATE_LABELS["B10"].endswith("1:32")
assert PROGRAMMED_VOLUME_DF.set_index("Well").loc["A10", "sample1"] == 6.25
assert PROGRAMMED_VOLUME_DF.set_index("Well").loc["B10", "sample2"] == 6.25

def highlight_volume_mismatches(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    delta_columns = [column for column in data.columns if column.startswith("Delta_")] + ["TotalDelta"]
    for column in delta_columns:
        styles.loc[data[column].abs() > 1e-9, column] = "background-color: #f8d7da"
    styles.loc[data["Match"], "Match"] = "background-color: #d8f3dc; font-weight: bold"
    styles.loc[~data["Match"], "Match"] = "background-color: #f8d7da; font-weight: bold"
    return styles

print("Expected well volumes (µL):")
display(EXPECTED_VOLUME_DF.replace(0, ""))
print("Programmed well volumes derived from PLATE_TRANSFER_GROUPS (µL):")
display(PROGRAMMED_VOLUME_DF.replace(0, ""))
print("Programmed minus expected comparison (µL):")
display(VOLUME_COMPARISON_DF.style.apply(highlight_volume_mismatches, axis=None))

ax = PROGRAMMED_VOLUME_DF.set_index("Well")[COMPONENT_COLUMNS].plot(
    kind="bar", stacked=True, figsize=(18, 7), width=0.82
)
ax.plot(
    range(len(EXPECTED_VOLUME_DF)), EXPECTED_VOLUME_DF["TOTALVOL(ul)"],
    color="black", marker="_", linewidth=1.5, label="Expected total (200 µL)",
)
ax.set_title("Programmed well composition vs. expected total")
ax.set_xlabel("Well")
ax.set_ylabel("Volume (µL)")
ax.set_ylim(0, 220)
ax.legend(ncol=3, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Deck construction

The bare `hamilton_tube_carrier_32_a00` at rail 19 holds all twelve 12 × 75 mm tubes without inserts. Five catalog 188042 DWP modules are installed on the catalog 188039 carrier. Only module 0 contains a plate.


In [ ]:
backend = LiquidHandlerChatterboxBackend(num_channels=8) if SIMULATE else STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tips_1000 = hamilton_96_tiprack_1000uL_filter(name="tips_1000ul_filter")
tips_300 = hamilton_96_tiprack_300uL_filter(name="tips_300ul_filter")
tips_50 = hamilton_96_tiprack_50uL_filter(name="tips_50ul_filter")
tip_carrier[0].assign_child_resource(tips_1000)
tip_carrier[1].assign_child_resource(tips_300)
tip_carrier[2].assign_child_resource(tips_50)
lh.deck.assign_child_resource(tip_carrier, rails=25)

tube_carrier_19 = hamilton_tube_carrier_32_a00(name="smp_car_32_rail19")
tube_resources = {}

def assign_12x75_tube(carrier, rail: int, key: str, position: int, initial_ul: float):
    tube = McKesson_tube_5mL_Rb(name=f"rail{rail}_{key}_position{position}")
    # The catalog resource omits this measured value, but STAR liquid handling requires it
    # to convert liquid heights from the tube cavity bottom to absolute deck Z.
    tube._material_z_thickness = TUBE_MATERIAL_Z_THICKNESS_MM
    holder = carrier[physical_to_site(position)]
    tube_bottom_in_carrier_mm = carrier.get_size_z() + TUBE_TOP_ABOVE_CARRIER_MM - tube.get_size_z()
    location = Coordinate(
        (holder.get_size_x() - tube.get_size_x()) / 2,
        (holder.get_size_y() - tube.get_size_y()) / 2,
        tube_bottom_in_carrier_mm - holder.location.z,
    )
    holder.assign_child_resource(tube, location=location)
    tube.set_volume(initial_ul)
    tube_resources[key] = tube
    return tube

for key, cfg in RAIL19_TUBES.items():
    assign_12x75_tube(tube_carrier_19, 19, key, cfg["position"], cfg["initial_ul"])
lh.deck.assign_child_resource(tube_carrier_19, rails=19)

mfx_modules = {i: hamilton_mfx_plateholder_DWP_metal_tapped(name=f"rail7_dwp_module_{i}") for i in range(5)}
mfx_carrier = hamilton_mfx_carrier_L5_base(name="mfx_carrier_188039_rail7", modules=mfx_modules)
assay_plate = cor_96_wellplate_360uL_Fb(name="black_endotoxin_assay_plate")
mfx_modules[0].assign_child_resource(assay_plate)
lh.deck.assign_child_resource(mfx_carrier, rails=7)

assert tube_carrier_19.model == "hamilton_tube_carrier_32_a00"
assert tube_carrier_19[0].children[0] is tube_resources["efw_standard"]
assert tube_carrier_19[physical_to_site(20)].children[0] is tube_resources["sample_1"]
assert tube_carrier_19[physical_to_site(19)].children[0] is tube_resources["sample_2"]
assert tube_carrier_19[physical_to_site(24)].children == []
assert all(tube.get_size_x() == 12 and tube.get_size_y() == 12 and tube.get_size_z() == 75 for tube in tube_resources.values())
assert all(tube.max_volume == 5000 for tube in tube_resources.values())
assert all(tube.material_z_thickness == TUBE_MATERIAL_Z_THICKNESS_MM for tube in tube_resources.values())
carrier_top_z = tube_carrier_19.get_absolute_location(z="t").z
assert all(abs(tube.get_absolute_location(z="t").z - carrier_top_z - TUBE_TOP_ABOVE_CARRIER_MM) < 1e-6 for tube in tube_resources.values())
assert tips_1000["A1"][0].get_tip().nominal_volume == 1000 and tips_1000["A1"][0].get_tip().has_filter
assert tips_300["A1"][0].get_tip().nominal_volume == 300 and tips_300["A1"][0].get_tip().has_filter
assert tips_50["A1"][0].get_tip().nominal_volume == 50 and tips_50["A1"][0].get_tip().has_filter
assert assay_plate.get_item("A1").max_volume == 360

print(lh.deck.summary())


## Connect and define execution helpers

Confirm the physical deck exactly matches the printed summary. Standard preparation consumes EFW from position 32; plate dilution uses EFW from position 31. Positions 30 and 29 remain reserves.


In [ ]:
if not lh.setup_finished:
    if SIMULATE:
        await lh.setup()
    else:
        await lh.setup(skip_autoload=True)

protocol_state = {"standards_prepared": False, "working_dispensed": False, "plate_loaded": False}
attached_channels = set()

async def confirm(message: str):
    if SIMULATE:
        print(f"[SIMULATION CONFIRMED] {message}")
        return
    response = input(f"\n{message} Type GO and press Enter to continue: " ).strip().upper()
    if response != "GO":
        raise RuntimeError("Operator cancelled the protocol.")

def plate_well(name: str):
    return assay_plate.get_item(name)

def plate_dispense_height(name: str) -> float:
    well = plate_well(name)
    current_ul = well.tracker.get_used_volume()
    if current_ul <= 0:
        return 1.0
    return min(well.compute_height_from_volume(current_ul) + 1.0, well.get_size_z() - 1.0)

async def pickup(tip_rack, positions, channels):
    spots = [tip_rack.get_item(position) for position in positions]
    await lh.pick_up_tips(spots, use_channels=channels)
    attached_channels.update(channels)

async def discard(channels):
    await lh.discard_tips(use_channels=channels)
    attached_channels.difference_update(channels)

def tube_aspiration_geometry(tube, aspiration_ul: float):
    """Return explicit surface height and following distance for a 12 × 75 mm tube."""
    current_ul = tube.tracker.get_used_volume()
    if current_ul < aspiration_ul:
        raise RuntimeError(f"{tube.name} has {current_ul:.1f} µL tracked; cannot aspirate {aspiration_ul:.1f} µL.")
    current_height_mm = tube.compute_height_from_volume(current_ul)
    final_height_mm = tube.compute_height_from_volume(current_ul - aspiration_ul)
    surface_following_mm = current_height_mm - final_height_mm
    return current_ul, current_height_mm, surface_following_mm

async def add_efw_to_standard_tubes():
    efw_tip_sizes = {required_tip_size(step["efw_ul"]) for step in STANDARD_PREP}
    if efw_tip_sizes != {1000}:
        raise RuntimeError(f"Standard EFW steps violate the 1000 µL tip policy: {efw_tip_sizes}")
    await pickup(tip_rack_for_volume(STANDARD_PREP[0]["efw_ul"]), ["A1"], [SINGLE_CHANNEL])
    efw_tube = tube_resources["efw_standard"]
    for step in STANDARD_PREP:
        volume_ul = step["efw_ul"]
        current_ul, liquid_height_mm, surface_following_mm = tube_aspiration_geometry(efw_tube, volume_ul)
        print(
            f"EFW standard: aspirating {volume_ul:.1f} µL from {current_ul:.1f} µL; "
            f"liquid_height={liquid_height_mm:.2f} mm, surface_following={surface_following_mm:.2f} mm."
        )
        await lh.aspirate(
            [efw_tube], 
            vols=[volume_ul],
            use_channels=[SINGLE_CHANNEL],
            liquid_height=[liquid_height_mm],
            surface_following_distance=[surface_following_mm],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            lld_search_height=[TUBE_LLD_SEARCH_HEIGHT_MM],
            immersion_depth=[2.0],
        )
        await lh.dispense(
            [tube_resources[step["key"]]],
            vols=[volume_ul],
            use_channels=[SINGLE_CHANNEL],
            liquid_height=[TUBE_DISPENSE_HEIGHT_MM],
            blow_out=[True],
            settling_time=[1])
    await discard([SINGLE_CHANNEL])

async def mix_standard(standard_key: str, tip_position: str):
    await pickup(tip_rack_for_volume(STANDARD_MIX_VOLUME_UL), [tip_position], [SINGLE_CHANNEL])
    for cycle in range(1, STANDARD_MIX_CYCLES + 1):
        await lh.aspirate([tube_resources[standard_key]], vols=[STANDARD_MIX_VOLUME_UL], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_ASPIRATE_HEIGHT_MM], flow_rates=[STANDARD_MIX_FLOW_UL_S])
        await lh.dispense(
            [tube_resources[standard_key]], vols=[STANDARD_MIX_VOLUME_UL], use_channels=[SINGLE_CHANNEL],
            # liquid_height=[6.0], flow_rates=[400], # changed to reach lower for more mixing
            liquid_height=[3.0], flow_rates=[400],
            mix=[Mix(volume=STANDARD_MIX_VOLUME_UL, repetitions=STANDARD_MIX_REPETITIONS, flow_rate=STANDARD_MIX_FLOW_UL_S)],
            blow_out=[True], settling_time=[1],
        )
        print(f"{standard_key}: completed mix/pause cycle {cycle}/{STANDARD_MIX_CYCLES}.")
        await asyncio.sleep(STANDARD_PAUSE_SECONDS)
    await discard([SINGLE_CHANNEL])

async def serial_transfer(source_key: str, destination_key: str, tip_position: str):
    transfer_volume_ul = 100.0
    await pickup(tip_rack_for_volume(transfer_volume_ul), [tip_position], [SINGLE_CHANNEL])
    await lh.aspirate([tube_resources[source_key]], vols=[transfer_volume_ul], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_ASPIRATE_HEIGHT_MM])
    await lh.dispense([tube_resources[destination_key]], vols=[transfer_volume_ul], use_channels=[SINGLE_CHANNEL], liquid_height=[TUBE_DISPENSE_HEIGHT_MM], blow_out=[True], settling_time=[1])
    await discard([SINGLE_CHANNEL])

def tip_rack_for_volume(volume_ul: float):
    return {50: tips_50, 300: tips_300, 1000: tips_1000}[required_tip_size(volume_ul)]

async def dispense_source_batch(group):
    """Aspirate once, dispense to assigned wells, and discard with reserve remaining."""
    source = tube_resources[group["source"]]
    dispense_total_ul = sum(volume_ul for _, volume_ul in group["destinations"])
    reserve_ul = group["reserve_ul"]
    aspiration_ul = dispense_total_ul + reserve_ul
    if not group.get("batch_aspirate", False) or reserve_ul <= 0.0:
        raise RuntimeError(f"Plate transfer must be a reserved single-aspiration batch: {group}")
    if aspiration_ul > group["tip_size"] or len(group["destinations"]) > 4:
        raise RuntimeError(f"Plate transfer batch exceeds its limits: {group}")
    rack = {50: tips_50, 300: tips_300, 1000: tips_1000}[group["tip_size"]]
    await pickup(rack, [group["tip"]], [SINGLE_CHANNEL])
    print(
        f"{group['source']} batch with {group['tip_size']} µL tip {group['tip']}: "
        f"aspirating {aspiration_ul:.1f} µL once to dispense "
        f"{dispense_total_ul:.1f} µL and discard {reserve_ul:.1f} µL reserve; "
        f"source={source.tracker.get_used_volume():.1f} µL."
    )
    if group["source"] == "efw_plate":
        _, liquid_height_mm, surface_following_mm = tube_aspiration_geometry(source, aspiration_ul)
        await lh.aspirate(
            [source], vols=[aspiration_ul], use_channels=[SINGLE_CHANNEL],
            liquid_height=[liquid_height_mm],
            surface_following_distance=[surface_following_mm],
            lld_mode=[STARBackend.LLDMode.GAMMA],
            lld_search_height=[TUBE_LLD_SEARCH_HEIGHT_MM],
            immersion_depth=[2.0], transport_air_volume=[0],
        )
    else:
        await lh.aspirate(
            [source], vols=[aspiration_ul], use_channels=[SINGLE_CHANNEL],
            liquid_height=[PLATE_SOURCE_ASPIRATE_HEIGHT_MM],
            transport_air_volume=[0],
        )
    for destination, volume_ul in group["destinations"]:
        await lh.dispense(
            [plate_well(destination)],
            vols=[volume_ul],
            use_channels=[SINGLE_CHANNEL],
            liquid_height=[plate_dispense_height(destination)],
            transport_air_volume=[0],
            blow_out=[False],
            settling_time=[1],
        )
    # The tip has contacted working solution. Discard it with the reserve still inside.
    await discard([SINGLE_CHANNEL])

async def dispense_group(group):
    await dispense_source_batch(group)

async def recover_attached_tips():
    if not attached_channels:
        print("No tips are recorded as attached.")
        return
    channels = sorted(attached_channels)
    await confirm(f"Recovery will discard tips from physical channels {[c + 1 for c in channels]}. Verify it is safe to move.")
    await discard(channels)
    print("Attached-tip recovery complete.")

async def reset_working_solution_iteration(source_volume_ul: float = 2000.0):
    """Reset software tracking after physically restoring the working-solution setup."""
    if attached_channels:
        raise RuntimeError("Tips are still recorded as attached. Run `await recover_attached_tips()` first.")
    if not 0.0 < source_volume_ul <= 5000.0:
        raise ValueError("source_volume_ul must be between 0 and 5000 µL.")
    await confirm(
        f"Tracking reset only: verify both working-solution tubes physically contain "
        f"{source_volume_ul:.1f} µL, all 30 assay wells are physically empty or the plate "
        "has been replaced, channels 5-6 have no tips attached, and fresh 1000 µL "
        "tips are physically present at F1 and G1."
    )
    tube_resources["working_1"].set_volume(source_volume_ul)
    tube_resources["working_2"].set_volume(source_volume_ul)
    for well_name in WORKING_WELLS:
        plate_well(well_name).set_volume(0.0)
    tips_1000.set_tip_state({"F1": True, "G1": True})
    protocol_state["working_dispensed"] = False
    protocol_state["plate_loaded"] = False
    print(
        f"Working-solution tracking reset: sources={source_volume_ul:.1f} µL each, "
        "30 assay wells=0 µL, tips F1/G1=present. No robot motion was performed."
    )

def required_plate_source_volumes():
    required = defaultdict(float)
    for group in PLATE_TRANSFER_GROUPS:
        required[group["source"]] += sum(volume_ul for _, volume_ul in group["destinations"])
        required[group["source"]] += group.get("reserve_ul", 0.0)
    return dict(required)

def validate_plate_loading_tracking():
    insufficient = {}
    for source_key, required_ul in required_plate_source_volumes().items():
        tracked_ul = tube_resources[source_key].tracker.get_used_volume()
        if tracked_ul + 1e-6 < required_ul:
            insufficient[source_key] = (tracked_ul, required_ul)
    if insufficient:
        details = ", ".join(
            f"{key}: tracked {tracked:.1f} µL, required {required:.1f} µL"
            for key, (tracked, required) in insufficient.items()
        )
        raise RuntimeError(
            f"Plate-loading source tracking is not ready ({details}). If the physical "
            "standards and plate were prepared outside this kernel, physically restore "
            "the complete plate-loading setup and run "
            "`await reset_assay_plate_loading_iteration()` before retrying."
        )
    unexpected_wells = {
        well_name: plate_well(well_name).tracker.get_used_volume()
        for well_name in WORKING_WELLS
        if abs(plate_well(well_name).tracker.get_used_volume() - 100.0) > 1e-6
    }
    if unexpected_wells:
        raise RuntimeError(
            "Plate-loading requires every assay well to start with exactly 100 µL of "
            f"working solution; mismatches: {unexpected_wells}."
        )

async def reset_assay_plate_loading_iteration(plate_volume_ul: float = 100.0):
    """Reset plate-loading tracking to an explicitly confirmed physical plate volume."""
    if attached_channels:
        raise RuntimeError("Tips are still recorded as attached. Run `await recover_attached_tips()` first.")
    if abs(plate_volume_ul) < 1e-6:
        plate_start_ul = 0.0
        working_solution_is_present = False
        plate_description = "empty; dispense_working_solution() will run next"
    elif abs(plate_volume_ul - 100.0) < 1e-6:
        plate_start_ul = 100.0
        working_solution_is_present = True
        plate_description = "at 100 µL working solution per used well"
    else:
        raise ValueError(
            f"plate_volume_ul must be 0 or 100 µL for this workflow, got {plate_volume_ul}."
        )
    await confirm(
        f"Tracking reset only: verify the assay plate is {plate_description}; "
        "EFW plate tube has 4000 µL; sample tubes have 1000 µL; "
        "standards at positions 25-28 contain 900, 900, 900, and 1000 µL respectively; "
        "and every filtered tip named in PLATE_TRANSFER_GROUPS is physically present."
    )
    source_start_ul = {
        "efw_standard": 550.0,
        "efw_plate": 4000.0, "sample_1": 1000.0, "sample_2": 1000.0,
        "std_5": 900.0, "std_0_5": 900.0, "std_0_05": 900.0,
        "std_0_005": 1000.0,
    }
    for source_key, volume_ul in source_start_ul.items():
        tube_resources[source_key].set_volume(volume_ul)
    if working_solution_is_present:
        tube_resources["working_1"].set_volume(500.0)
        tube_resources["working_2"].set_volume(500.0)
    for well_name in WORKING_WELLS:
        plate_well(well_name).set_volume(plate_start_ul)
    racks_by_size = {50: tips_50, 300: tips_300, 1000: tips_1000}
    tips_by_size = defaultdict(dict)
    for group in PLATE_TRANSFER_GROUPS:
        tips_by_size[group["tip_size"]][group["tip"]] = True
    for tip_size, tip_states in tips_by_size.items():
        racks_by_size[tip_size].set_tip_state(tip_states)
    protocol_state["standards_prepared"] = True
    protocol_state["working_dispensed"] = working_solution_is_present
    protocol_state["plate_loaded"] = False
    if working_solution_is_present:
        validate_plate_loading_tracking()
    print(
        f"Assay plate-loading tracking reset with plate wells at {plate_start_ul:.1f} µL; "
        f"working_dispensed={working_solution_is_present}. No robot motion was performed."
    )

async def reset_repeat_workflow(source_volume_ul: float = 2000.0):
    """Atomically reset tracking for repeating working solution and plate loading."""
    await reset_working_solution_iteration(source_volume_ul=source_volume_ul)
    await reset_assay_plate_loading_iteration(plate_volume_ul=0.0)
    if protocol_state["working_dispensed"] or protocol_state["plate_loaded"]:
        raise RuntimeError(f"Repeat reset left invalid protocol state: {protocol_state}")
    insufficient = {
        source_key: (tube_resources[source_key].tracker.get_used_volume(), required_ul)
        for source_key, required_ul in required_plate_source_volumes().items()
        if tube_resources[source_key].tracker.get_used_volume() + 1e-6 < required_ul
    }
    if insufficient:
        raise RuntimeError(f"Repeat reset left insufficient plate-loading sources: {insufficient}")
    print(
        "Repeat workflow ready: empty plate, working sources reset, plate-loading "
        "sources reset, working_dispensed=False, plate_loaded=False."
    )


In [ ]:
# await pickup(tips_1000, ["A1"], channels=[0])

In [ ]:

# await lh.aspirate(
#     [tube_resources["efw_standard"]],
#     vols=[900],
#     use_channels=[0],
#     liquid_height = [56.1],
#     # lld_mode=[STARBackend.LLDMode.GAMMA],
#     # auto_surface_following_distance=[True],
#     # probe_liquid_height=[True],
#     # lld_search_height=[TUBE_LLD_SEARCH_HEIGHT_MM],
#     # immersion_depth=[2.0],
# )

In [ ]:
# await lh.dispense(
#     [tube_resources["efw_standard"]],
#     vols=[900],
#     use_channels=[0],
#     liquid_height = [55.1],
#     swap_speed = [160],
#     # lld_mode=[STARBackend.LLDMode.GAMMA],
#     # auto_surface_following_distance=[True],
#     # probe_liquid_height=[False],
#     # lld_search_height=[TUBE_LLD_SEARCH_HEIGHT_MM],
#     # immersion_depth=[2.0],
# )

## Protocol stages

The stages are independently callable but enforce their required order. `run_protocol()` performs all three. If a stage fails after motion begins, do not rerun it blindly; inspect the physical deck and use the recovery cell if a tip remains attached.


In [ ]:
async def prepare_standards():
    if protocol_state["standards_prepared"]:
        raise RuntimeError("Standards are already marked prepared.")
    await confirm("Verify rail 19 positions 25-28 are empty standard tubes, position 32 contains 4.0 mL EFW, and all specified filtered tips are present.")
    await add_efw_to_standard_tubes()
    await confirm("Add exactly 250 µL of resolubilized 20 EU/mL endotoxin stock to physical position 25 (the tube containing 750 µL EFW).")
    tube_resources["std_5"].set_volume(1000.0)
    await mix_standard("std_5", STANDARD_PREP[0]["mix_tip"])
    for step in STANDARD_PREP[1:]:
        await serial_transfer(step["from"], step["key"], step["transfer_tip"])
        await mix_standard(step["key"], step["mix_tip"])
    protocol_state["standards_prepared"] = True
    print("Standards prepared: positions 25-28 contain 5, 0.5, 0.05, and 0.005 EU/mL, respectively.")

async def dispense_working_solution():
    # if not protocol_state["standards_prepared"]:
    #     raise RuntimeError("Prepare standards before dispensing working solution.")
    if protocol_state["working_dispensed"]:
        raise RuntimeError("Working solution is already marked dispensed.")
    await confirm("Verify 2 mL fresh working solution is present in rail 19 physical positions 23 and 22. Working solution will be added first, contrary to the kit guide.")
    maximum_batch_aspiration_ul = 6 * 100.0 + 100.0
    await pickup(tip_rack_for_volume(maximum_batch_aspiration_ul), ["F1", "G1"], WORKING_CHANNELS)
    sources = [tube_resources["working_1"], tube_resources["working_2"]]
    channel_wells = [
        [f"A{column}" for column in range(1, 13)] + ["C1", "D1", "E1"],
        [f"B{column}" for column in range(1, 13)] + ["C2", "D2", "E2"],
    ]
    batch_size = 6
    dispense_volume_ul = 100.0 # I agree this should be 100 but errs; 
    return_volume_ul = 100.0
    for start_index in range(0, len(channel_wells[0]), batch_size):
        batches = [wells[start_index:start_index + batch_size] for wells in channel_wells]
        aspirate_volumes = [len(batch) * dispense_volume_ul + return_volume_ul for batch in batches]
        if any(required_tip_size(volume_ul) != 1000 for volume_ul in aspirate_volumes):
            raise RuntimeError(f"Working-solution batch violates the 1000 µL tip policy: {aspirate_volumes}")
        aspiration_geometries = [
            tube_aspiration_geometry(source, volume_ul)
            for source, volume_ul in zip(sources, aspirate_volumes)
        ]
        source_heights = [geometry[1] for geometry in aspiration_geometries]
        surface_following_distances = [geometry[2] for geometry in aspiration_geometries]
        print(
            f"Working solution batch {start_index // batch_size + 1}: "
            f"aspirating {aspirate_volumes[0]:.1f} µL per source at "
            f"heights {[round(height, 2) for height in source_heights]} mm."
        )
        await lh.aspirate(
            sources,
            vols=aspirate_volumes,
            use_channels=WORKING_CHANNELS,
            liquid_height=[height - 4.0 for height in source_heights],  # Aspirate 4 mm lower.
            surface_following_distance=surface_following_distances,
            transport_air_volume=[0,0]
            )
        for dispense_index, destination_names in enumerate(zip(*batches)):
            destinations = [plate_well(name) for name in destination_names]
            await lh.dispense(
                destinations,
                vols=[100.0, 100.0],
                use_channels=WORKING_CHANNELS,
                liquid_height=[4.0, 4.0],
                blow_out=[False, False],
                settling_time=[1, 1],
                transport_air_volume=[0,0],
            )
        # Return up to the extra 100 µL reserve to the working-solution source tubes.
        pending_return_indices = list(range(len(WORKING_CHANNELS)))
        attempted_return_ul = return_volume_ul
        returned_by_channel = {}
        last_return_error = None
        while pending_return_indices and attempted_return_ul > 0:
            attempt_sources = [sources[index] for index in pending_return_indices]
            attempt_channels = [WORKING_CHANNELS[index] for index in pending_return_indices]
            attempt_heights = [source.compute_height_from_volume(source.tracker.get_used_volume()) for source in attempt_sources]
            try:
                await lh.dispense(
                    attempt_sources,
                    vols=[attempted_return_ul] * len(attempt_sources),
                    use_channels=attempt_channels,
                    liquid_height=attempt_heights,
                    transport_air_volume=[0] * len(attempt_sources),
                    blow_out=[True] * len(attempt_sources),
                )
            except Exception as error:
                last_return_error = error
                channel_errors = getattr(error, "errors", None)
                if channel_errors is not None:
                    failed_channels = set(channel_errors)
                    succeeded_indices = [index for index in pending_return_indices if WORKING_CHANNELS[index] not in failed_channels]
                    for index in succeeded_indices:
                        returned_by_channel[WORKING_CHANNELS[index]] = attempted_return_ul
                    pending_return_indices = [index for index in pending_return_indices if WORKING_CHANNELS[index] in failed_channels]
                attempted_return_ul -= 5.0
                if pending_return_indices:
                    print(
                        f"Return-to-source dispense failed for channels "
                        f"{[channel + 1 for channel in [WORKING_CHANNELS[index] for index in pending_return_indices]]}; "
                        f"retrying with {attempted_return_ul:.1f} µL. Error: {error}"
                    )
            else:
                for index in pending_return_indices:
                    returned_by_channel[WORKING_CHANNELS[index]] = attempted_return_ul
                pending_return_indices = []
        if pending_return_indices:
            raise RuntimeError("Could not return working-solution reserve to source tubes.") from last_return_error
        print(
            "Returned working-solution reserve: "
            + ", ".join(
                f"channel {channel + 1} -> {volume_ul:.1f} µL"
                for channel, volume_ul in sorted(returned_by_channel.items())
            )
        )
        
    await discard(WORKING_CHANNELS)
    protocol_state["working_dispensed"] = True
    print("Working solution dispensed to all 30 wells; 500 µL remains in each source tube.")

async def load_assay_plate():
    # if not protocol_state["standards_prepared"] or not protocol_state["working_dispensed"]:
    #     raise RuntimeError("Prepare standards and dispense working solution before loading the remaining plate components.")
    if protocol_state["plate_loaded"]:
        raise RuntimeError("Plate is already marked loaded. Reset only after physically restoring the setup.")
    validate_plate_loading_tracking()
    for group in PLATE_TRANSFER_GROUPS:
        await dispense_group(group)
        print(f"Completed {group['source']} transfers with {group['tip_size']} µL filtered tips.")
    for well_name in WORKING_WELLS:
        actual = plate_well(well_name).tracker.get_used_volume()
        if abs(actual - 200.0) > 1e-6:
            raise RuntimeError(f"{well_name} volume mismatch: expected 200 µL, tracked {actual} µL.")
    protocol_state["plate_loaded"] = True
    print("Assay plate loading complete: all 30 used wells contain 200 µL.")

def runtime_volume_report():
    report = {key: tube.tracker.get_used_volume() for key, tube in tube_resources.items()}
    if protocol_state["plate_loaded"]:
        for key, expected in PLAN_CHECK["expected_residuals_ul"].items():
            if abs(report[key] - expected) > 1e-6:
                raise RuntimeError(f"{key} residual mismatch: expected {expected} µL, tracked {report[key]} µL.")
    return report




## Run

Uncomment the call below only after checking the deck summary and preflight report. The three stage functions may instead be run individually.


In [ ]:
async def run_protocol():
    try:
        await prepare_standards()
        await dispense_working_solution()
        await load_assay_plate()
    except Exception:
        if attached_channels:
            print(f"Protocol stopped with tips recorded on physical channels {[c + 1 for c in sorted(attached_channels)]}. Inspect the deck before recovery.")
        raise
    report = runtime_volume_report()
    print("Protocol complete. Tracked source residuals (µL):")
    for source, volume in report.items():
        print(f"  {source:16s} {volume:8.2f}")
    return report

result = await run_protocol()


## Recovery

Run only after an interrupted stage reports attached tips and after confirming that movement to the waste is safe. This does not repair liquid volumes or make a partially completed stage safe to rerun.


In [ ]:
# await recover_attached_tips()
# await reset_repeat_workflow(source_volume_ul=2000.0)
# await reset_working_solution_iteration(source_volume_ul=2000.0)
await reset_assay_plate_loading_iteration(plate_volume_ul=100.0) # Plate-loading requires every assay well to start with exactly 100 µL
# await lh.discard_tips()
# sources = [tube_resources["working_1"], tube_resources["working_2"]]

# await lh.dispense(
#     sources,
#     vols=[10.0, 10.0],
#     use_channels=[4, 5],
#     liquid_height=[20.0, 20.0],
#     blow_out=[True, True],
# )
# await lh.drop_tips(tips_1000["F1:G1"], use_channels=[4, 5])

## Stop

Run only after the protocol or recovery is complete.


In [ ]:
# await lh.stop()
